In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)
n = 2000

age = np.random.randint(21, 65, n)
income = np.random.randint(20000, 150001, n)
credit_score = np.random.randint(300, 851, n)
loan_term = np.random.choice([12, 24, 36, 48, 60], n)
employment_years = np.random.randint(0, 31, n)

previous_defaults = np.random.choice(
    [0, 1, 2, 3], n, p=[0.72, 0.18, 0.07, 0.03]
)

employment_type = np.random.choice(
    ["Salaried", "Self-employed", "Business", "Contract"], n
)

marital_status = np.random.choice(
    ["Single", "Married", "Divorced"], n
)

education = np.random.choice(
    ["High School", "Graduate", "Postgraduate"], n
)

application_date = pd.to_datetime(
    np.random.choice(
        pd.date_range("2022-01-01", "2025-12-31"), n
    )
)

loan_amount = (
    income * np.random.uniform(0.25, 1.15, n)
    + (credit_score - 300) * 35
    + loan_term * 180
    + np.random.normal(0, 5000, n)
)

loan_amount = np.clip(loan_amount, 5000, 200000)

risk_score = (
    -0.0035 * (credit_score - 600)
    + 0.000012 * (loan_amount - 50000)
    + 0.55 * previous_defaults
    - 0.025 * employment_years
    + np.random.normal(0, 0.45, n)
)

default_probability = 1 / (1 + np.exp(-risk_score))
loan_default = np.random.binomial(1, default_probability)

data = pd.DataFrame({
    "age": age,
    "income": income,
    "credit_score": credit_score,
    "loan_amount": loan_amount,
    "loan_term": loan_term,
    "employment_years": employment_years,
    "previous_defaults": previous_defaults,
    "employment_type": employment_type,
    "marital_status": marital_status,
    "education": education,
    "application_date": application_date,
    "loan_default": loan_default
})

for column in [
    "income",
    "credit_score",
    "employment_years",
    "employment_type"
]:
    missing_indices = data.sample(
        frac=0.03, random_state=42
    ).index

    data.loc[missing_indices, column] = np.nan

TARGET = "loan_default"

X = data.drop(columns=[TARGET]).copy()
y = data[TARGET].copy()

print("DATASET INFORMATION")
print("-" * 60)

print("Dataset shape:", data.shape)
print("Number of records:", len(data))
print("Number of input features:", X.shape[1])

print("\nFirst five rows:")
print(data.head())

print("\nMissing values:")
print(data.isnull().sum())

print("\nTarget distribution:")
print(y.value_counts())

print("\nDATA SPLITTING")
print("-" * 60)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training data:", X_train.shape)
print("Validation data:", X_val.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nValidation target distribution:")
print(y_val.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

print("\nTest set is kept aside.")

print("\nNUMERIC AND CATEGORICAL COLUMNS")
print("-" * 60)

numeric_columns = [
    "age",
    "income",
    "credit_score",
    "loan_amount",
    "loan_term",
    "employment_years",
    "previous_defaults",
    "application_month",
    "application_year",
    "income_loan_interaction",
    "log_loan_amount"
]

categorical_columns = [
    "employment_type",
    "marital_status"
]

ordinal_columns = ["education"]

print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)
print("Ordinal columns:", ordinal_columns)

print("\nSIMPLE IMPUTATION")
print("-" * 60)

raw_numeric_columns = [
    "age",
    "income",
    "credit_score",
    "loan_amount",
    "loan_term",
    "employment_years",
    "previous_defaults"
]

X_train_numeric = X_train[raw_numeric_columns]

simple_imputer = SimpleImputer(strategy="median")

simple_imputed = simple_imputer.fit_transform(
    X_train_numeric
)

print("Original numeric data shape:", X_train_numeric.shape)
print("After SimpleImputer:", simple_imputed.shape)
print(
    "Missing values after SimpleImputer:",
    np.isnan(simple_imputed).sum()
)

print("\nKNN IMPUTATION")
print("-" * 60)

knn_imputer = KNNImputer(n_neighbors=5)

knn_imputed = knn_imputer.fit_transform(
    X_train_numeric
)

print("Original numeric data shape:", X_train_numeric.shape)
print("After KNNImputer:", knn_imputed.shape)
print(
    "Missing values after KNNImputer:",
    np.isnan(knn_imputed).sum()
)

print("\nFEATURE ENGINEERING")
print("-" * 60)

def create_features(df):
    df = df.copy()

    df["application_month"] = pd.to_datetime(
        df["application_date"]
    ).dt.month

    df["application_year"] = pd.to_datetime(
        df["application_date"]
    ).dt.year

    df["income_loan_interaction"] = (
        df["income"] * df["loan_amount"]
    )

    df["log_loan_amount"] = np.log1p(
        df["loan_amount"].clip(lower=0)
    )

    df = df.drop(columns=["application_date"])

    return df

X_train_features = create_features(X_train)

print("Original feature count:", X_train.shape[1])
print("After feature engineering:", X_train_features.shape[1])

print("\nNew engineered features:")
print(
    X_train_features[
        [
            "application_month",
            "application_year",
            "income_loan_interaction",
            "log_loan_amount"
        ]
    ].head()
)

print("\nCOLUMN TRANSFORMER")
print("-" * 60)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

ordinal_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[
            ["High School", "Graduate", "Postgraduate"]
        ],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns),
        ("ordinal", ordinal_pipeline, ordinal_columns)
    ]
)

print("Numeric pipeline: Median Imputer + StandardScaler")
print("Categorical pipeline: Most Frequent Imputer + OneHotEncoder")
print("Ordinal pipeline: Most Frequent Imputer + OrdinalEncoder")
print("ColumnTransformer created successfully.")

print("\nCOMPLETE MACHINE LEARNING PIPELINE")
print("-" * 60)

model_pipeline = Pipeline(steps=[
    ("feature_engineering", FunctionTransformer(
        create_features,
        validate=False
    )),
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=3000))
])

print(model_pipeline)

print("\n5-FOLD CROSS-VALIDATION")
print("-" * 60)

cv_scores = cross_val_score(
    model_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("5-fold CV scores:", np.round(cv_scores, 4))
print("CV mean accuracy: {:.4f}".format(cv_scores.mean()))
print("CV standard deviation: {:.4f}".format(cv_scores.std()))

print(
    "Final CV result: {:.4f} +/- {:.4f}".format(
        cv_scores.mean(),
        cv_scores.std()
    )
)

print("\nMODEL TRAINING")
print("-" * 60)

model_pipeline.fit(X_train, y_train)

train_predictions = model_pipeline.predict(X_train)
train_accuracy = accuracy_score(y_train, train_predictions)

print("Model training completed.")
print("Training accuracy: {:.4f}".format(train_accuracy))

print("\nVALIDATION")
print("-" * 60)

val_predictions = model_pipeline.predict(X_val)
val_accuracy = accuracy_score(y_val, val_predictions)

print("Validation accuracy: {:.4f}".format(val_accuracy))

print("\nSAVE MODEL USING JOBLIB")
print("-" * 60)

model_filename = "task2_pipeline.pkl"

joblib.dump(model_pipeline, model_filename)

print("Model saved:", model_filename)
print("File exists:", os.path.exists(model_filename))

print("\nLOAD SAVED MODEL")
print("-" * 60)

loaded_model = joblib.load(model_filename)

print("Model loaded successfully.")
print("Loaded model type:", type(loaded_model))

print("\nPREDICT THREE NEW ROWS")
print("-" * 60)

new_rows = X_val.head(3).copy()

new_predictions = loaded_model.predict(new_rows)
new_probabilities = loaded_model.predict_proba(new_rows)

prediction_results = pd.DataFrame({
    "Actual": y_val.head(3).values,
    "Predicted": new_predictions,
    "Default Probability": new_probabilities[:, 1]
})

print(prediction_results.to_string(index=False))

print("\nMODEL PERFORMANCE")
print("-" * 60)

print("Cross-validation mean accuracy:", round(cv_scores.mean(), 4))
print("Training accuracy:", round(train_accuracy, 4))
print("Validation accuracy:", round(val_accuracy, 4))



DATASET INFORMATION
------------------------------------------------------------
Dataset shape: (2000, 12)
Number of records: 2000
Number of input features: 11

First five rows:
   age    income  credit_score   loan_amount  loan_term  employment_years  \
0   59   45426.0         527.0  40134.134469         24              28.0   
1   49   37772.0         401.0  44205.463282         36               2.0   
2   35  123754.0         716.0  61620.350552         24              25.0   
3   63   23712.0         515.0  25648.428084         60              13.0   
4   28   21367.0         706.0  39032.375343         36              24.0   

   previous_defaults employment_type marital_status    education  \
0                  0        Contract       Divorced  High School   
1                  0        Salaried       Divorced  High School   
2                  0        Business       Divorced  High School   
3                  2        Salaried        Married     Graduate   
4                  